# A forward pass, from one layer to a batch

A colleague trained a small network that sorts seeds into three varieties from two measurements
and sent over its weights as plain arrays. Here you write the code that runs them: the shapes, one
layer, the softmax, the full forward pass with its cache, the loss, and the accuracy on a batch of
180 seeds. The lesson's by-hand example is the test for each step of the forward pass,
and the colleague's weights test the shapes, the loss and the accuracy.

**How to work through it.** Each part asks for one function or class. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Each seed has two measurements, length and width, already rescaled the way the training data
were. Each row of `X` is one seed, and `y` holds its variety as 0, 1 or 2. The weights are stored
the way the lesson's code expects, with `W1` of shape `(2, 5)` for a hidden layer of five ReLU
units and `W2` of shape `(5, 3)` for the three outputs. Run this cell first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

rng = np.random.default_rng(0)
centers = np.array([[-1.4, 0.0], [1.0, 1.0], [0.8, -1.2]])          # one cluster per variety
X = np.concatenate([c + rng.normal(0, 0.65, (60, 2)) for c in centers])
y = np.repeat([0, 1, 2], 60)
order = rng.permutation(len(y))                                       # shuffle the rows
X, y = X[order], y[order]

# The weights the colleague sent, rounded to two decimals.
W1 = np.array([[0.22, -1.72, 2.50, 0.51, 0.36],
               [1.10, 0.36, 0.49, -1.49, -1.86]])
b1 = np.array([0.31, 0.51, 0.96, -0.95, -0.44])
W2 = np.array([[-0.37, 0.48, -1.41],
               [0.73, -0.68, -1.02],
               [-2.08, 1.20, 0.68],
               [0.32, -0.90, 1.60],
               [0.13, -0.97, 1.10]])
b2 = np.array([0.23, -0.41, 0.18])
params = [(W1, b1), (W2, b2)]


def relu(Z):
    return np.maximum(0.0, Z)


plt.scatter(X[:, 0], X[:, 1], c=y, cmap="viridis", s=18)
plt.xlabel("length (rescaled)")
plt.ylabel("width (rescaled)")
plt.show()
print("X", X.shape, " y", y.shape)
print(X[:3])
print(y[:3])

## Part 1 (coding task): The widths of the network

Before any arithmetic, read the shapes. In the rows-are-examples layout, layer $l$'s weight
matrix has shape $n_{l-1} \times n_l$: one row per input to the layer and one column per unit in
it. Its bias has one entry per unit. The output of one layer is the input of the next, so the
number of columns of `W1` must equal the number of rows of `W2`. A forward pass that runs but
returns nonsense is almost always a shape error, so check the shapes once and fail loudly.

Write `widths(params)`. For `params = [(W1, b1), (W2, b2), ...]` it returns the list of layer
widths $[n_0, n_1, \dots, n_L]$. It raises a `ValueError` if a bias does not have one entry per
column of its weight matrix, or if two consecutive layers do not fit together.

In [ ]:
def widths(params):
    """[n_0, n_1, ..., n_L] from the weight shapes. Raises ValueError if the layers do not fit."""
    # TODO
    raise NotImplementedError

In [ ]:
assert list(widths(params)) == [2, 5, 3], f"the colleague's network is 2 -> 5 -> 3, got {widths(params)}"
assert list(widths([(np.zeros((4, 7)), np.zeros(7))])) == [4, 7], "one layer with 4 inputs and 7 units gives [4, 7]"
for bad, why in [
    ([(np.zeros((2, 5)), np.zeros(5)), (np.zeros((4, 3)), np.zeros(3))], "5 hidden units cannot feed a layer that expects 4 inputs"),
    ([(np.zeros((2, 5)), np.zeros(3))], "a bias of 3 entries does not match 5 units"),
]:
    try:
        widths(bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f"widths() should raise ValueError because {why}")
print("Part 1 passed")

## Part 2 (coding task): One layer's weighted sums

One layer computes every unit's weighted sum for every example at once, $Z = A W + \mathbf{b}$.
`A` has one example per row, `A @ W` gives one row of weighted sums per example, and the bias, one
row of length $n_l$, is added to every row. NumPy does that last part on its own, which is called
broadcasting.

Write `linear(A, W, b)`, which returns $Z$ with shape `(N, n_l)`. The check runs it on the lesson's
first layer, where the input $(1, 2)$ gave the weighted sums $(-1, 2, 1)$.

In [ ]:
def linear(A, W, b):
    """Weighted sums A @ W + b, one row per example."""
    # TODO
    raise NotImplementedError

In [ ]:
# The lesson's first layer, stored transposed for the rows-are-examples layout.
W1_lesson = np.array([[1.0, 0.0, 1.0], [-1.0, 1.0, 1.0]])
b1_lesson = np.array([0.0, 0.0, -2.0])
Z = linear(np.array([[1.0, 2.0], [0.0, 1.0]]), W1_lesson, b1_lesson)
assert Z.shape == (2, 3), f"two examples through three units should give shape (2, 3), got {Z.shape}"
assert np.allclose(Z[0], [-1.0, 2.0, 1.0]), f"the lesson's z1 is (-1, 2, 1), got {Z[0]}"
assert np.allclose(Z[1], [-1.0, 1.0, -1.0]), f"the second row should be (-1, 1, -1), with the bias added to it too, got {Z[1]}"
assert linear(X, W1, b1).shape == (180, 5), "the 180 seeds through the colleague's first layer should give (180, 5)"
print("Part 2 passed")

## Part 3 (coding task): Softmax, written so it cannot overflow

The output layer turns each row of logits into probabilities,
$\operatorname{softmax}(\mathbf{z})_i = e^{z_i} / \sum_j e^{z_j}$. Typed in as written it
overflows, because a logit of 1,000 makes $e^{1000}$ infinite and the ratio `nan`. Subtracting
each row's largest logit from every entry in that row changes nothing, since the factor cancels top
and bottom, and leaves every exponent at or below 0.

Write `softmax(Z)`. It works on a whole batch at once, one row at a time, with the shift. The check
includes the lesson's logits $(1, 3)$, which give $(0.1192, 0.8808)$, and a row with a logit of
1,000.

In [ ]:
def softmax(Z):
    """Row-wise softmax of a batch of logits, with the shift that prevents overflow."""
    # TODO
    raise NotImplementedError

In [ ]:
P = softmax(np.array([[1.0, 3.0]]))
assert P.shape == (1, 2), f"one row of two logits should give shape (1, 2), got {P.shape}"
assert np.allclose(P, [[0.1192, 0.8808]], atol=1e-4), f"softmax of (1, 3) is (0.1192, 0.8808), got {P}"
P = softmax(np.array([[1.0, 3.0], [2.0, 2.0], [1000.0, 0.0]]))
assert np.allclose(P.sum(axis=1), 1.0), "every row should sum to 1"
assert np.allclose(P[1], [0.5, 0.5]), f"equal logits should give equal probabilities, got {P[1]}"
assert np.all(np.isfinite(P)) and np.allclose(P[2], [1.0, 0.0]), f"a logit of 1000 should give (1, 0), not nan, got {P[2]}"
assert np.allclose(softmax(np.array([[1.0, 3.0]])), softmax(np.array([[101.0, 103.0]]))), "adding the same number to every logit in a row changes nothing"
print("Part 3 passed")

## Part 4 (coding task): The whole forward pass, with its cache

Now chain the pieces: `linear`, then ReLU, then `linear` again, then `softmax`. Keep every
intermediate array on the way, because backpropagation needs each layer's pre-activation $Z$ to
differentiate the activation and each layer's input $A$ to differentiate the matrix multiply. The
lesson stores them in a dictionary keyed `"A0"`, `"Z1"`, `"A1"`, `"Z2"`, `"A2"`.

Write `forward(X, params)`. It returns `(probs, cache)` for a two-layer network with ReLU in the
hidden layer and softmax at the output. The check runs the lesson's example through it and expects
every number from the hand computation: $\mathbf{z}^{(1)} = (-1, 2, 1)$,
$\mathbf{a}^{(1)} = (0, 2, 1)$, $\mathbf{z}^{(2)} = (1, 3)$ and
$\hat{\mathbf{y}} = (0.1192, 0.8808)$.

In [ ]:
def forward(X, params):
    """Output probabilities (N, n_L) and the cache {"A0", "Z1", "A1", "Z2", "A2"}."""
    # TODO
    raise NotImplementedError

In [ ]:
lesson = [(W1_lesson, b1_lesson),
          (np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]), np.array([0.0, 0.0]))]
probs, cache = forward(np.array([[1.0, 2.0]]), lesson)
assert isinstance(cache, dict) and {"A0", "Z1", "A1", "Z2", "A2"} <= set(cache), f"the cache should hold A0, Z1, A1, Z2 and A2, got {sorted(cache)}"
assert np.allclose(cache["Z1"], [[-1.0, 2.0, 1.0]]), f"Z1 should be (-1, 2, 1), got {cache['Z1']}"
assert np.allclose(cache["A1"], [[0.0, 2.0, 1.0]]), f"A1 should be (0, 2, 1), since ReLU turns the first unit off, got {cache['A1']}"
assert np.allclose(cache["Z2"], [[1.0, 3.0]]), f"the logits should be (1, 3), got {cache['Z2']}"
assert np.allclose(probs, [[0.1192, 0.8808]], atol=1e-4), f"the output should be (0.1192, 0.8808), got {probs}"
assert np.array_equal(cache["A2"], probs), "A2 is the output itself"

probs, cache = forward(X, params)
assert probs.shape == (180, 3), f"180 seeds and 3 varieties should give (180, 3), got {probs.shape}"
assert cache["Z1"].shape == (180, 5) and cache["A1"].shape == (180, 5), "Z1 and A1 should be (180, 5)"
assert np.allclose(probs.sum(axis=1), 1.0), "each row of probabilities should sum to 1"
for name in ("A0", "Z1", "A1", "Z2", "A2"):
    print(name, cache[name].shape)
print("Part 4 passed")

## Part 5 (coding task): The cross-entropy loss

With labels as integers, the cross-entropy of one example is $-\log$ of the probability the
network gave its true class, and the loss of a batch is the mean over the rows. The lesson's
example put probability 0.1192 on the true class 0, so its loss is $-\log 0.1192 = 2.127$. A coin
flip between two classes would score $\log 2 = 0.693$, and certainty on the right class would
score 0.

Write `cross_entropy(probs, y)`. `probs` has one row per example and `y` holds one integer label
per row. `probs[np.arange(N), y]` picks each row's probability of its own label. Return the mean
of the negative logs as a float.

In [ ]:
def cross_entropy(probs, y):
    """Mean of -log(probability of the true class) over the rows, as a float."""
    # TODO
    raise NotImplementedError

In [ ]:
loss = cross_entropy(np.array([[0.1192, 0.8808]]), np.array([0]))
assert isinstance(loss, float), f"the loss should be a Python float, got {type(loss).__name__}"
assert abs(loss - 2.127) < 1e-3, f"-log(0.1192) = 2.127, got {loss:.4f}"
assert abs(cross_entropy(np.array([[0.5, 0.5], [0.5, 0.5]]), np.array([0, 1])) - np.log(2)) < 1e-9, "coin flips should score log 2 = 0.693"
assert abs(cross_entropy(np.array([[0.2, 0.8], [0.9, 0.1]]), np.array([1, 0])) + (np.log(0.8) + np.log(0.9)) / 2) < 1e-9, "the batch loss is the mean over rows, each row picking its own label"
batch_loss = cross_entropy(forward(X, params)[0], y)
assert batch_loss < 0.3, f"a trained network should score well below log 3 = 1.099 here, got {batch_loss:.4f}"
print(f"loss of the colleague's network on the 180 seeds: {batch_loss:.4f}")
print("Part 5 passed")

## Part 6 (coding task): Predictions and accuracy

The network's prediction for a row is the class with the highest probability, the position of
the largest entry in that row. Accuracy is the fraction of rows where that position equals the
label. This is what tells you whether the colleague's weights, run through your code, actually
sort seeds.

Write `accuracy(probs, y)`, which returns that fraction as a float. The check draws the network's
decision regions by running a grid of points through `forward` and coloring each by its
prediction, with the seeds on top.

In [ ]:
def accuracy(probs, y):
    """Fraction of rows whose highest-probability class equals the label, as a float."""
    # TODO
    raise NotImplementedError

In [ ]:
four = np.array([[0.2, 0.7, 0.1], [0.6, 0.3, 0.1], [0.1, 0.1, 0.8], [0.5, 0.4, 0.1], [0.4, 0.4, 0.2]])
assert accuracy(four, np.array([1, 0, 2, 2, 0])) == 0.8, "four of these five rows are right, and a tie counts for the first class"
acc = accuracy(forward(X, params)[0], y)
assert isinstance(acc, float), f"accuracy should be a Python float, got {type(acc).__name__}"
assert acc > 0.9, f"the colleague's network should sort over 90% of the seeds correctly, got {acc:.3f}"

g1, g2 = np.meshgrid(np.linspace(-4, 4, 200), np.linspace(-4, 4, 200))
grid = np.column_stack([g1.ravel(), g2.ravel()])
regions = forward(grid, params)[0].argmax(axis=1).reshape(g1.shape)
plt.contourf(g1, g2, regions, levels=[-0.5, 0.5, 1.5, 2.5], cmap="viridis", alpha=0.25)
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="viridis", s=18)
plt.xlabel("length (rescaled)")
plt.ylabel("width (rescaled)")
plt.title(f"accuracy {acc:.3f}")
plt.show()
print("Part 6 passed")

## Solution

### Part 1: The widths of the network

In [ ]:
def widths(params):
    """[n_0, n_1, ..., n_L] from the weight shapes. Raises ValueError if the layers do not fit."""
    out = [params[0][0].shape[0]]
    for W, b in params:
        n_in, n_out = W.shape
        if b.shape != (n_out,):
            raise ValueError(f"a bias of shape {b.shape} does not match a weight matrix with {n_out} columns")
        if n_in != out[-1]:
            raise ValueError(f"a layer with {n_in} inputs follows one with {out[-1]} outputs")
        out.append(n_out)
    return out

### Part 2: One layer's weighted sums

In [ ]:
def linear(A, W, b):
    """Weighted sums A @ W + b, one row per example."""
    return A @ W + b

### Part 3: Softmax, written so it cannot overflow

In [ ]:
def softmax(Z):
    """Row-wise softmax of a batch of logits, with the shift that prevents overflow."""
    Z = Z - Z.max(axis=1, keepdims=True)
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)

### Part 4: The whole forward pass, with its cache

In [ ]:
def forward(X, params):
    """Output probabilities (N, n_L) and the cache {"A0", "Z1", "A1", "Z2", "A2"}."""
    (W1, b1), (W2, b2) = params
    cache = {"A0": X}
    cache["Z1"] = linear(X, W1, b1)
    cache["A1"] = relu(cache["Z1"])
    cache["Z2"] = linear(cache["A1"], W2, b2)
    cache["A2"] = softmax(cache["Z2"])
    return cache["A2"], cache

### Part 5: The cross-entropy loss

In [ ]:
def cross_entropy(probs, y):
    """Mean of -log(probability of the true class) over the rows, as a float."""
    picked = probs[np.arange(len(y)), y]
    return float(-np.mean(np.log(picked)))

### Part 6: Predictions and accuracy

In [ ]:
def accuracy(probs, y):
    """Fraction of rows whose highest-probability class equals the label, as a float."""
    return float(np.mean(probs.argmax(axis=1) == y))